# L24 · 链式法则（chain rule） — 函数套函数的求导，反向传播（backpropagation，BP）的数学基础

**目标**：复合函数 `f(g(x))` 的导数（derivative） = `f'(g(x)) · g'(x)`——一层层连乘。

**为什么对 Aurora 重要**：Month 2 的 `autograd（automatic differentiation，autograd）` 引擎在 `backward()` 里沿计算图（computation graph）连乘偏导（partial derivative，∂f/∂x）——`composite_derivative` 的分拆方式直接对应计算图上每条边的局部梯度（local gradient）。

← **上一课**　[L23 · 梯度](L23_gradients.ipynb)

> 上节课学习了 **梯度**：多元函数的"最陡上坡"方向，偏导与梯度向量的计算。  
> 本课将探讨 **链式法则**。

## 本课剧情：追踪变化如何传递

复合函数 `f(g(x))` 的变化是两层变化相乘的结果：内层把 `x` 的扰动缩放一次，外层再对中间结果做同样的事。链式法则把这两步形式化为 `dy/dx = f'(g(x)) · g'(x)`——本节用数值差分验证这个等式，再把它推广到任意深度的函数嵌套。

## 1. 例子：y = sin(x²)

外层 f(u)=sin(u)，内层 g(x)=x²。
`dy/dx = cos(x²) · 2x`

链式法则描述的是变化如何一层层传递：x 变化 Δx，内层 u=x² 随之变化约 g'(x)·Δx=2x·Δx；u 的这个变化再驱动 y=sin(u) 变化约 f'(u)·(2x·Δx)=cos(x²)·2x·Δx。两步合并得 dy/dx=cos(x²)·2x。神经网络的反向传播就是这个过程的推广——每层缓存局部斜率，梯度从损失端沿计算图逐层连乘传回权重。

## 实验入口：用数值变化观察函数

下面在 `x = 1.3` 处计算 `y = sin(x²)` 的数值导数，并与解析值 `cos(x²)·2x` 对比。关注 `numerical` 和 `analytical` 两列的误差大小。

In [ ]:
import numpy as np
x = 1.3
analytic = np.cos(x**2) * 2*x          # 链式法则
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
print('链式法则:', round(analytic,5), ' 数值验证:', round(numeric,5))

> **[热身格 — 非链式法则]** 下面两格用最简单的多项式（`f(x)=x²`、`f(x)=x²+2x`）演练「数值差分」技术本身，
> 不涉及函数嵌套，因此**不是链式法则的应用**。目的是让你在接触复合函数前先确认差分公式的精度。
> 真正的链式法则示例从下一节（2. 实现 `composite_derivative`）开始。

## 动手观察：变化率不是一句口号

在一组 `x` 值上同时打印函数值和近似斜率，观察数值差分在每个点的精度。

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

下面遍历一列 `x`，把函数值和解析导数并排输出，确认链式法则在每个点都与数值差分吻合。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `composite_derivative(x)` —— y=sin(x²) 的导数

**推理路线**：
1. 拆层：外层 f(u)=sin(u)，内层 g(x)=x²，链式法则给出 dy/dx = f'(g(x)) · g'(x)
2. 分别求导：f'(u)=cos(u)，代入 u=x² 得 cos(x²)；g'(x)=2x
3. 相乘：结果是 cos(x²)·2x，NumPy 广播自动保证对数组输入也正确

**参考输入输出**：x=√(π/2) 时，x²=π/2，dy/dx=cos(π/2)·2x=0·√(2π)=0（sin(x²) 在该点取极值，导数为零）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `composite_derivative` 前明确三件事：
- 输入：`x`（标量或数组，单位弧度）
- 关键步骤：先算内层 `g(x) = x²`，再把外层导数 `cos(x²)` 和内层导数 `2x` 相乘
- 返回：与 `x` 等形状的导数数组，可直接和数值差分对比

In [ ]:
def composite_derivative(x):
    # ✏️ TODO: sin(x^2) 对 x 的导数
    # 提示：外层 f(u)=sin(u) → f'(u)=cos(u)；内层 g(x)=x² → g'(x)=2x
    # 链式法则：dy/dx = cos(x²) · 2x
    raise NotImplementedError("TODO: 实现 composite_derivative — 返回 cos(x**2) * 2*x")


In [ ]:
x = 1.3
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
try:
    result = composite_derivative(x)
    assert abs(result - numeric) < 1e-6, (
        f"误差 {abs(result - numeric):.2e} 过大（应 < 1e-6，即 1μs 级精度）"
    )
    print("✅ 通过：你手算了一次链式法则 = 一次最小的反向传播。")
except NotImplementedError:
    print("⚠️  composite_derivative 尚未实现，请先完成 TODO 再运行此格。")


**🔗 Aurora 连接**：第 L 层输出 → 第 L-1 层 → ... → 权重，梯度沿这条链连乘传回去，就是 Month 2 你要从零实现的 autograd。

## 参数实验：3 层复合的链式法则

构造 3 层复合 h(x)=sin(exp(x²))：
- 最内层 u=x²，导数 du/dx=2x
- 中间层 v=exp(u)，导数 dv/du=exp(u)=exp(x²)
- 外层 y=sin(v)，导数 dy/dv=cos(v)=cos(exp(x²))

链式法则给出：dh/dx = cos(exp(x²)) · exp(x²) · 2x

用步长 `h=1e-7` 的数值差分在 x=[0, 1, -1] 处各算一次数值导数，与上式手算值对比，误差应在 1e-9 以内。这验证了链式法则可直接推广到任意深度的嵌套——autograd 计算图做的正是同一件事。

In [ ]:
# 参数实验：h(x) = sin(exp(x²)) 链式法则验证
# dh/dx = cos(exp(x²)) · exp(x²) · 2x
def h_analytic(x):
    return np.cos(np.exp(x**2)) * np.exp(x**2) * 2 * x

print(f"{'x':>6}  {'解析导数':>14}  {'数值差分':>14}  {'误差':>10}")
print('-' * 50)
for x in [0.0, 1.0, -1.0]:
    analytic = h_analytic(x)
    numerical = (np.sin(np.exp((x+1e-7)**2)) - np.sin(np.exp((x-1e-7)**2))) / 2e-7
    err = abs(analytic - numerical)
    print(f"{x:>6.1f}  {analytic:>14.8f}  {numerical:>14.8f}  {err:>10.2e}")
    assert err < 1e-9, f'x={x}: 误差 {err} 过大'
print('✅ sin(exp(x²)) 的链式法则在三点处均验证通过')


In [ ]:
# 扩展：4 层复合 g(x) = tanh(sin(x²+1))
# dg/dx = (1 - tanh²(sin(x²+1))) · cos(x²+1) · 2x
def g_analytic(x):
    u = x**2 + 1
    v = np.sin(u)
    w = np.tanh(v)
    return (1 - w**2) * np.cos(u) * 2 * x

x = 0.8
analytic = g_analytic(x)
numerical = (np.tanh(np.sin((x+1e-7)**2+1)) - np.tanh(np.sin((x-1e-7)**2+1))) / 2e-7
err = abs(analytic - numerical)
print(f'x={x}: 解析导数={analytic:.8f}, 数值差分={numerical:.8f}, 误差={err:.2e}')
assert err < 1e-5
print('✅ 4 层复合链式法则验证通过')
print('  autograd 做的正是这件事——只是参数空间是百万维，链更深')


## 本课收束

现在能调用 `composite_derivative(x)` 得到 `sin(x²)` 在任意点的导数，数值误差在 `1e-9` 量级内。这个"外层导数 × 内层导数"的结构对应 `autograd.backward()`：每个节点缓存局部导数，沿计算图向输入方向连乘。Month 2 实现 autograd 时，`composite_derivative` 的分拆方式会直接变成计算图的边和节点设计。

下一课：**L25**（梯度下降）把今天得到的导数当方向信号，把参数沿负梯度方向推动一步。

---

→ **下一课**　[L25 · 梯度下降](L25_gradient_descent.ipynb)

> 下节课将学习 **梯度下降**：用一条直线拟合数据，从损失函数到权重更新公式。